# Receipt Expense Agent: OCR, Field Extraction, Categorization & Reimbursement

An **agent pipeline** that processes expense receipts end-to-end:

1. **OCR** — extract raw text from receipt images (simulated)
2. **Field Extractor** — parse structured fields (vendor, date, amount, items, tax)
3. **Expense Categorizer** — classify each receipt into a company expense category
4. **Policy Validator** — check amounts against reimbursement policy limits
5. **Summary Generator** — produce a formatted reimbursement report

---

## Why "OCR + Agent Chaining"?

Traditional expense processing is a monolithic script. An **agent chain** decomposes it into independent, testable stages:

```text
Receipt Image
     │
     ▼
┌────────────┐    OCR converts pixels to text. In production: Tesseract,
│  OCR Agent  │    Google Vision, Azure Document Intelligence, or AWS Textract.
└─────┬──────┘    Here we simulate with pre-built receipt text strings.
      │ raw text
      ▼
┌────────────────┐    Regex + heuristic extraction. In production: an LLM
│ Field Extractor │    with structured-output mode (tool_choice / response_format).
└─────┬──────────┘
      │ ReceiptData (typed)
      ▼
┌──────────────────┐    Rule-based (or LLM) classification into company
│ Expense Categorizer│    expense categories (meals, travel, software, etc.)
└─────┬────────────┘
      │ categorized receipt
      ▼
┌──────────────────┐    Checks per-category limits, per-receipt caps,
│ Policy Validator  │    required fields, duplicate detection.
└─────┬────────────┘
      │ validated receipt + warnings
      ▼
┌──────────────────┐    Aggregates all receipts into a reimbursement
│ Summary Generator │    report with totals, policy flags, and approval status.
└──────────────────┘
```

### Why Chain Instead of One Big Function?

| Benefit | Explanation |
|---|---|
| **Testability** | Test OCR parsing without needing categorization logic |
| **Replaceability** | Swap the OCR engine without touching downstream agents |
| **Observability** | Log each stage independently — see exactly where a receipt failed |
| **Reusability** | The categorizer can be reused for credit-card statement processing |
| **Parallelization** | OCR + extraction can run in parallel across receipts |


## 1. Setup

In [ ]:
!pip install -q pandas matplotlib seaborn

In [ ]:
import json
import re
import textwrap
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from datetime import date, datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)

ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## 2. Data Model

Every stage of the pipeline produces a **typed dataclass**. This makes the data flow traceable and every intermediate result inspectable.


In [ ]:
class ExpenseCategory(Enum):
    MEALS = "meals"
    TRAVEL = "travel"
    LODGING = "lodging"
    SOFTWARE = "software"
    OFFICE_SUPPLIES = "office_supplies"
    CONFERENCE = "conference"
    COMMUNICATION = "communication"
    TRANSPORTATION = "transportation"
    CLIENT_ENTERTAINMENT = "client_entertainment"
    MISCELLANEOUS = "miscellaneous"

    def __str__(self):
        return self.value


class PolicyStatus(Enum):
    APPROVED = "approved"
    WARNING = "warning"          # Over soft limit but under hard limit
    REJECTED = "rejected"        # Over hard limit or missing required field
    DUPLICATE = "duplicate"

    def __str__(self):
        return self.value


@dataclass
class LineItem:
    """A single item on a receipt."""
    description: str
    quantity: int
    unit_price: float
    total: float


@dataclass
class ReceiptData:
    """Structured data extracted from a receipt."""
    receipt_id: str
    vendor_name: str
    vendor_address: str
    receipt_date: str              # ISO format
    payment_method: str            # "visa", "amex", "cash", etc.
    currency: str
    subtotal: float
    tax: float
    tip: float
    total: float
    items: list[LineItem]
    raw_text: str                  # Original OCR output
    confidence: float              # OCR confidence 0-1


@dataclass
class CategorizedReceipt:
    """Receipt with expense category assigned."""
    receipt: ReceiptData
    category: ExpenseCategory
    category_confidence: float     # 0-1
    category_reason: str


@dataclass
class PolicyResult:
    """Result of policy validation."""
    status: PolicyStatus
    warnings: list[str]
    approved_amount: float         # May be capped
    original_amount: float
    policy_notes: str


@dataclass
class ProcessedReceipt:
    """Fully processed receipt — the final output of the agent chain."""
    receipt: ReceiptData
    category: ExpenseCategory
    category_confidence: float
    policy: PolicyResult
    processed_at: str = ""

    def __post_init__(self):
        if not self.processed_at:
            self.processed_at = datetime.now().isoformat()


@dataclass
class ReimbursementReport:
    """Summary report for an expense submission."""
    employee_name: str
    submission_date: str
    receipts: list[ProcessedReceipt]
    total_submitted: float
    total_approved: float
    total_rejected: float
    category_totals: dict
    warnings: list[str]
    approval_status: str           # "auto_approved", "needs_review", "rejected"


print("Data model loaded.")
print(f"  Expense categories: {[c.value for c in ExpenseCategory]}")
print(f"  Policy statuses:    {[s.value for s in PolicyStatus]}")

## 3. Simulated Receipt Images → OCR Text

In production, the OCR agent would:
1. Accept an image (JPEG, PNG, or PDF scan)
2. Run through an OCR engine (Tesseract, Google Vision API, AWS Textract)
3. Output raw text with confidence scores and bounding boxes

Here we **simulate** this with pre-built receipt text strings that mimic real OCR output — including minor formatting noise.

### OCR Agent Design

```text
┌─────────────┐     image bytes      ┌──────────────┐
│ Receipt JPEG │ ──────────────────►  │   OCR Engine  │
└─────────────┘                      │  (Tesseract)  │
                                     └──────┬───────┘
                                            │ raw text + confidence
                                     ┌──────▼───────┐
                                     │  OCR Agent    │
                                     │  - denoise    │
                                     │  - deskew     │
                                     │  - normalize  │
                                     └──────┬───────┘
                                            │ cleaned text
                                            ▼
                                     str + float(confidence)
```


In [ ]:
# Each receipt simulates what OCR would extract from a scanned image
RECEIPT_TEXTS = {
    "REC-001": {
        "text": """
SAKURA JAPANESE RESTAURANT
123 Main Street, San Francisco, CA 94105
Tel: (415) 555-0123

Date: 06/12/2024
Server: Mike T.
Table: 14

-----------------------------------------
Miso Soup               2  x  $4.50   $9.00
Salmon Sashimi           1  x $18.95  $18.95
Chicken Teriyaki         1  x $15.50  $15.50
Edamame                  1  x  $5.00   $5.00
Green Tea                2  x  $2.50   $5.00
-----------------------------------------
Subtotal:                         $53.45
Tax (8.625%):                      $4.61
Tip:                              $10.00
-----------------------------------------
TOTAL:                            $68.06

Payment: Visa ****4832
Auth: 847291

Thank you for dining with us!
        """,
        "confidence": 0.96,
    },
    "REC-002": {
        "text": """
UBER TECHNOLOGIES
Receipt

Trip on Jun 10, 2024
Pick-up: 345 Market St, SF
Drop-off: SFO Airport Terminal 2

Distance: 14.2 mi
Duration: 28 min

Fare                           $32.50
Surge                           $0.00
Booking Fee                     $3.75
Airport Surcharge               $5.00
-----------------------------------------
Subtotal                       $41.25
Tax                             $3.41

Total                          $44.66

Payment: Amex ****9901

Ride ID: UBER-2024-0610-7842
        """,
        "confidence": 0.98,
    },
    "REC-003": {
        "text": """
MARRIOTT COURTYARD
800 Convention Way
Austin, TX 78701
(512) 555-8800

Guest: Sarah Johnson
Check-in:  06/09/2024
Check-out: 06/11/2024
Room: 412 - King Suite

Night 1 (Jun 9)               $189.00
Night 2 (Jun 10)              $189.00
Room Service Jun 10             $34.50
Parking (2 nights)              $30.00
-----------------------------------------
Subtotal:                      $442.50
Resort Fee:                     $25.00
Tax (15%):                      $70.13
-----------------------------------------
TOTAL:                         $537.63

Payment: Corporate Visa ****3344

Confirmation: MRT-2024-90125
        """,
        "confidence": 0.94,
    },
    "REC-004": {
        "text": """
GITHUB
88 Colin P Kelly Jr St
San Francisco, CA 94107

Invoice #GH-2024-58392
Date: June 1, 2024

GitHub Enterprise Cloud
Team Plan - 5 seats
Billing Period: Jun 2024

5 x $21.00/seat/month         $105.00
-----------------------------------------
Subtotal:                      $105.00
Tax:                             $0.00
-----------------------------------------
Total:                         $105.00

Payment: Visa ****4832
        """,
        "confidence": 0.99,
    },
    "REC-005": {
        "text": """
STAPLES
2100 El Camino Real
Palo Alto, CA 94306
(650) 555-2100

Date: 06/08/2024  Time: 14:32

Copy Paper (5 ream)     1  x  $34.99  $34.99
Printer Ink HP 61       2  x  $29.99  $59.98
Sticky Notes (12pk)     1  x   $8.49   $8.49
Pens BIC (24pk)         2  x   $6.99  $13.98
-----------------------------------------
Subtotal:                        $117.44
Tax (9.25%):                      $10.86
-----------------------------------------
TOTAL:                           $128.30

Payment: Corporate Amex ****9901
Trans ID: STP-84291
        """,
        "confidence": 0.95,
    },
    "REC-006": {
        "text": """
BLUE BOTTLE COFFEE
66 Mint Street
San Francisco, CA 94103

06/13/2024 8:15 AM

Latte Grande              1 x  $5.50   $5.50
Croissant                 1 x  $4.25   $4.25
Drip Coffee               1 x  $3.75   $3.75
-----------------------------------------
Subtotal:                         $13.50
Tax:                               $1.16
-----------------------------------------
Total:                            $14.66

Payment: Visa ****4832
        """,
        "confidence": 0.97,
    },
    "REC-007": {
        "text": """
PYCON US 2024
Pittsburgh Convention Center
1000 Fort Duquesne Blvd
Pittsburgh, PA 15222

REGISTRATION RECEIPT

Attendee: Sarah Johnson
Type: Professional
Early Bird

Registration Fee               $450.00
Tutorial Add-on (1 day)         $75.00
-----------------------------------------
Subtotal:                       $525.00
Processing Fee:                  $15.75
Tax:                              $0.00
-----------------------------------------
Total:                          $540.75

Payment: Visa ****4832
Conf #: PYCON-2024-SJ-4829
        """,
        "confidence": 0.93,
    },
    "REC-008": {
        "text": """
VERIZON WIRELESS
One Verizon Way
Basking Ridge, NJ 07920

Account: ***-***-5591
Billing Date: June 1, 2024

Monthly Plan (Unlimited)        $55.00
Device Payment                  $15.00
Insurance                        $7.00
-----------------------------------------
Subtotal:                        $77.00
Taxes & Fees:                     $8.47
-----------------------------------------
Total Due:                       $85.47

Auto-Pay: Amex ****9901
        """,
        "confidence": 0.97,
    },
    "REC-009": {
        "text": """
THE CAPITAL GRILLE
555 California Street
San Francisco, CA 94104
(415) 555-9000

Date: 06/11/2024
Guests: 4
Server: Andrea

Wagyu Steak              1 x  $85.00  $85.00
Lobster Tail             1 x  $62.00  $62.00
Filet Mignon             1 x  $58.00  $58.00
Sea Bass                 1 x  $45.00  $45.00
Wine Bottle (Opus One)   1 x $350.00 $350.00
Desserts                 4 x  $16.00  $64.00
-----------------------------------------
Subtotal:                        $664.00
Tax (8.625%):                     $57.27
Tip:                             $144.25
-----------------------------------------
TOTAL:                           $865.52

Payment: Corporate Visa ****3344

Client Dinner - Acme Corp Q3 Review
        """,
        "confidence": 0.92,
    },
    "REC-010": {
        "text": """
BART (Bay Area Rapid Transit)
Clipper Card Reload

Date: 06/12/2024
Station: Embarcadero

Reload Amount:                   $20.00
-----------------------------------------
Total Charged:                   $20.00

Payment: Visa ****4832
Card #: CLIP-****-8392
        """,
        "confidence": 0.99,
    },
}

print(f"Simulated receipts loaded: {len(RECEIPT_TEXTS)}")
for rid, data in RECEIPT_TEXTS.items():
    lines = [l.strip() for l in data["text"].strip().split("\n") if l.strip()]
    print(f"  {rid}: {lines[0][:35]:<35} conf={data['confidence']:.2f}  lines={len(lines)}")

## 4. Agent 1 — OCR Agent (Simulated)

In production, this agent would call an OCR API. Here it retrieves the pre-built text and adds realistic noise at lower confidence levels.


In [ ]:
class OCRAgent:
    """Simulates OCR processing of receipt images."""

    def __init__(self):
        self._call_log: list[dict] = []

    def process(self, receipt_id: str) -> tuple[str, float]:
        """Process a receipt image and return (raw_text, confidence)."""
        entry = RECEIPT_TEXTS.get(receipt_id)
        if not entry:
            self._call_log.append({"receipt_id": receipt_id, "success": False})
            return "", 0.0

        text = entry["text"].strip()
        confidence = entry["confidence"]

        # Simulate OCR noise at lower confidence
        if confidence < 0.95:
            # Occasionally introduce OCR-typical errors
            noise_map = {"l": "1", "O": "0", "S": "5"}
            chars = list(text)
            for i in range(len(chars)):
                if np.random.random() < 0.002 and chars[i] in noise_map:
                    chars[i] = noise_map[chars[i]]
            text = "".join(chars)

        self._call_log.append({
            "receipt_id": receipt_id,
            "success": True,
            "chars": len(text),
            "confidence": confidence,
        })
        return text, confidence

    @property
    def call_log(self):
        return self._call_log


ocr_agent = OCRAgent()

# Demo: process one receipt
demo_text, demo_conf = ocr_agent.process("REC-001")
print(f"OCR Confidence: {demo_conf:.2f}")
print(f"Characters: {len(demo_text)}")
print(f"First 200 chars:\n{demo_text[:200]}")

## 5. Agent 2 — Field Extractor

The field extractor uses **regex patterns** to parse structured fields from raw OCR text. In production, this would be an LLM with `response_format` set to a JSON schema.

### Extraction Strategy

| Field | Pattern | Fallback |
|---|---|---|
| Vendor name | First non-empty line | "Unknown Vendor" |
| Date | MM/DD/YYYY, Jun DD YYYY | Today |
| Items | `description  qty x $price $total` | Empty list |
| Subtotal | `subtotal: $XX.XX` | Sum of items |
| Tax | `tax: $XX.XX` | 0.00 |
| Tip | `tip: $XX.XX` | 0.00 |
| Total | `total: $XX.XX` | Subtotal + tax + tip |
| Payment | `visa/amex/cash ****NNNN` | "unknown" |


In [ ]:
class FieldExtractor:
    """Agent 2: Extract structured fields from OCR text."""

    # Regex patterns for common receipt fields
    DATE_PATTERNS = [
        r"(\d{2}/\d{2}/\d{4})",                          # MM/DD/YYYY
        r"((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2},?\s+\d{4})",  # Month DD, YYYY
        r"(\d{4}-\d{2}-\d{2})",                           # YYYY-MM-DD
        r"(?:Date|Trip on)[:\s]+(.+?)(?:\n|$)",            # "Date: ..." or "Trip on ..."
    ]

    AMOUNT_PATTERN = r"\$\s*([\d,]+\.?\d*)"
    ITEM_PATTERN = r"([A-Za-z][\w\s\(\)]+?)\s+(\d+)\s*x\s*\$?\s*([\d.]+)\s+\$?([\d.]+)"
    PAYMENT_PATTERN = r"((?:visa|amex|mastercard|corporate\s+\w+|cash|auto.pay)[\s:]*(?:\*{2,4})?\s*(\d{4})?)"

    def __init__(self):
        self._call_log: list[dict] = []

    def _find_date(self, text: str) -> str:
        for pattern in self.DATE_PATTERNS:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                raw = m.group(1).strip().rstrip(",")
                # Normalize to ISO format
                for fmt in ["%m/%d/%Y", "%B %d %Y", "%b %d %Y", "%Y-%m-%d",
                            "%B %d, %Y", "%b %d, %Y", "%b. %d, %Y"]:
                    try:
                        return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
                    except ValueError:
                        continue
                # Try partial matches
                m2 = re.search(r"(\w+)\s+(\d{1,2}),?\s+(\d{4})", raw)
                if m2:
                    try:
                        return datetime.strptime(
                            f"{m2.group(1)} {m2.group(2)} {m2.group(3)}", "%B %d %Y"
                        ).strftime("%Y-%m-%d")
                    except ValueError:
                        pass
                # raw date, return as-is
                return raw
        return date.today().isoformat()

    def _find_amount(self, text: str, label: str) -> float:
        pattern = rf"{label}\s*[:\s]*\$?\s*([\d,]+\.?\d*)"
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return float(m.group(1).replace(",", ""))
        return 0.0

    def _find_total(self, text: str) -> float:
        # Look for the last/largest TOTAL
        patterns = [r"(?:TOTAL|Total Due|Total Charged)\s*[:\s]*\$?\s*([\d,]+\.?\d*)"]
        for pat in patterns:
            matches = re.findall(pat, text, re.IGNORECASE)
            if matches:
                return max(float(m.replace(",", "")) for m in matches)
        return 0.0

    def _find_items(self, text: str) -> list[LineItem]:
        items = []
        for m in re.finditer(self.ITEM_PATTERN, text):
            desc = m.group(1).strip()
            qty = int(m.group(2))
            unit = float(m.group(3))
            total = float(m.group(4))
            items.append(LineItem(desc, qty, unit, total))
        return items

    def _find_payment(self, text: str) -> str:
        m = re.search(self.PAYMENT_PATTERN, text, re.IGNORECASE)
        if m:
            raw = m.group(0).strip().lower()
            if "visa" in raw:
                card = re.search(r"\d{4}", raw)
                return f"visa ****{card.group()}" if card else "visa"
            if "amex" in raw:
                card = re.search(r"\d{4}", raw)
                return f"amex ****{card.group()}" if card else "amex"
            if "cash" in raw:
                return "cash"
            return raw[:30]
        return "unknown"

    def _find_vendor(self, text: str) -> tuple[str, str]:
        lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
        vendor = lines[0] if lines else "Unknown Vendor"
        # Address: look for a line with street number or city/state
        address = ""
        for line in lines[1:5]:
            if re.search(r"\d+\s+\w+\s+(st|street|ave|way|blvd|rd|real)", line, re.IGNORECASE):
                address = line
                break
        return vendor, address

    def extract(self, receipt_id: str, raw_text: str,
                confidence: float) -> ReceiptData:
        """Extract structured fields from raw OCR text."""
        vendor, address = self._find_vendor(raw_text)
        receipt_date = self._find_date(raw_text)
        items = self._find_items(raw_text)
        subtotal = self._find_amount(raw_text, "subtotal")
        tax_val = self._find_amount(raw_text, "tax")
        tip_val = self._find_amount(raw_text, "tip")
        total = self._find_total(raw_text)
        payment = self._find_payment(raw_text)

        # Fallbacks
        if subtotal == 0 and items:
            subtotal = sum(i.total for i in items)
        if total == 0:
            total = subtotal + tax_val + tip_val

        result = ReceiptData(
            receipt_id=receipt_id,
            vendor_name=vendor,
            vendor_address=address,
            receipt_date=receipt_date,
            payment_method=payment,
            currency="USD",
            subtotal=round(subtotal, 2),
            tax=round(tax_val, 2),
            tip=round(tip_val, 2),
            total=round(total, 2),
            items=items,
            raw_text=raw_text,
            confidence=confidence,
        )

        self._call_log.append({
            "receipt_id": receipt_id,
            "vendor": vendor,
            "total": total,
            "items_found": len(items),
            "confidence": confidence,
        })
        return result

    @property
    def call_log(self):
        return self._call_log


extractor = FieldExtractor()

# Demo
demo_receipt = extractor.extract("REC-001", demo_text, demo_conf)
print(f"Vendor:  {demo_receipt.vendor_name}")
print(f"Date:    {demo_receipt.receipt_date}")
print(f"Items:   {len(demo_receipt.items)}")
for item in demo_receipt.items:
    print(f"  {item.description:<25} {item.quantity} x ${item.unit_price:>6.2f} = ${item.total:.2f}")
print(f"Subtotal: ${demo_receipt.subtotal:.2f}")
print(f"Tax:      ${demo_receipt.tax:.2f}")
print(f"Tip:      ${demo_receipt.tip:.2f}")
print(f"Total:    ${demo_receipt.total:.2f}")
print(f"Payment:  {demo_receipt.payment_method}")

## 6. Agent 3 — Expense Categorizer

The categorizer uses **keyword rules** to assign each receipt to an expense category. In production, an LLM would handle ambiguous cases.

### Categorization Rules

| Category | Keyword Signals |
|---|---|
| Meals | restaurant, coffee, diner, bistro, food keywords |
| Travel | uber, lyft, airline, flight |
| Lodging | hotel, marriott, hilton, airbnb |
| Software | github, atlassian, aws, subscription |
| Office supplies | staples, paper, ink, pens |
| Conference | pycon, registration, conference, summit |
| Communication | verizon, at&t, phone, internet |
| Transportation | bart, muni, transit, parking |
| Client entertainment | client, dinner with, entertainment |


In [ ]:
# Keyword-based categorization rules
CATEGORY_RULES: list[tuple[ExpenseCategory, list[str], float]] = [
    (ExpenseCategory.CLIENT_ENTERTAINMENT,
     ["client", "client dinner", "client lunch", "entertainment", "acme corp", "review"],
     0.90),
    (ExpenseCategory.LODGING,
     ["marriott", "hilton", "hyatt", "hotel", "airbnb", "check-in", "check-out", "night", "room", "suite"],
     0.92),
    (ExpenseCategory.CONFERENCE,
     ["pycon", "conference", "summit", "registration", "attendee", "tutorial", "workshop"],
     0.93),
    (ExpenseCategory.SOFTWARE,
     ["github", "atlassian", "jira", "aws", "azure", "slack", "notion", "saas", "seats", "subscription",
      "invoice"],
     0.95),
    (ExpenseCategory.TRAVEL,
     ["uber", "lyft", "airline", "flight", "trip", "fare", "surge", "booking fee", "airport"],
     0.91),
    (ExpenseCategory.TRANSPORTATION,
     ["bart", "muni", "transit", "clipper", "metro", "bus", "rail", "reload"],
     0.94),
    (ExpenseCategory.COMMUNICATION,
     ["verizon", "at&t", "t-mobile", "wireless", "phone", "monthly plan", "internet"],
     0.93),
    (ExpenseCategory.OFFICE_SUPPLIES,
     ["staples", "office depot", "paper", "ink", "pens", "printer", "copy paper", "sticky notes",
      "toner", "supplies"],
     0.92),
    (ExpenseCategory.MEALS,
     ["restaurant", "cafe", "coffee", "diner", "bistro", "grill", "sushi", "pizza", "bar",
      "server:", "table:", "teriyaki", "sashimi", "latte", "croissant", "drip coffee",
      "japanese", "italian", "mexican", "thai"],
     0.88),
]


class ExpenseCategorizer:
    """Agent 3: Classify receipts into expense categories."""

    def __init__(self):
        self._call_log: list[dict] = []

    def categorize(self, receipt: ReceiptData) -> CategorizedReceipt:
        """Assign an expense category based on vendor and item keywords."""
        text_lower = (receipt.raw_text + " " + receipt.vendor_name).lower()

        best_cat = ExpenseCategory.MISCELLANEOUS
        best_score = 0.0
        best_reason = "No matching keywords found"
        best_conf = 0.50

        for category, keywords, base_conf in CATEGORY_RULES:
            hits = [kw for kw in keywords if kw in text_lower]
            if hits:
                score = len(hits) / len(keywords)
                if score > best_score:
                    best_score = score
                    best_cat = category
                    best_conf = min(base_conf * (1 + score), 0.99)
                    best_reason = f"Matched keywords: {', '.join(hits[:4])}"

        result = CategorizedReceipt(
            receipt=receipt,
            category=best_cat,
            category_confidence=round(best_conf, 3),
            category_reason=best_reason,
        )
        self._call_log.append({
            "receipt_id": receipt.receipt_id,
            "category": str(best_cat),
            "confidence": best_conf,
            "reason": best_reason[:60],
        })
        return result

    @property
    def call_log(self):
        return self._call_log


categorizer = ExpenseCategorizer()

# Demo
demo_cat = categorizer.categorize(demo_receipt)
print(f"Receipt:    {demo_cat.receipt.receipt_id} — {demo_cat.receipt.vendor_name}")
print(f"Category:   {demo_cat.category}")
print(f"Confidence: {demo_cat.category_confidence:.2f}")
print(f"Reason:     {demo_cat.category_reason}")

## 7. Agent 4 — Policy Validator

Company expense policies define **per-category limits** and **per-receipt caps**. The validator checks each receipt against these rules.


In [ ]:
# Company expense policy
EXPENSE_POLICY = {
    ExpenseCategory.MEALS: {
        "per_receipt_limit": 75.00,
        "daily_limit": 150.00,
        "requires_attendees": False,
        "notes": "Max $75 per meal. Tips included.",
    },
    ExpenseCategory.TRAVEL: {
        "per_receipt_limit": 200.00,
        "daily_limit": 400.00,
        "requires_attendees": False,
        "notes": "Economy class only. Pre-approval for flights > $500.",
    },
    ExpenseCategory.LODGING: {
        "per_receipt_limit": 250.00,      # Per night
        "daily_limit": 250.00,
        "requires_attendees": False,
        "notes": "Max $250/night. Book through corporate travel portal when possible.",
    },
    ExpenseCategory.SOFTWARE: {
        "per_receipt_limit": 500.00,
        "daily_limit": 500.00,
        "requires_attendees": False,
        "notes": "Pre-approval required for subscriptions > $200/month.",
    },
    ExpenseCategory.OFFICE_SUPPLIES: {
        "per_receipt_limit": 200.00,
        "daily_limit": 200.00,
        "requires_attendees": False,
        "notes": "Use corporate vendor catalog for orders > $100.",
    },
    ExpenseCategory.CONFERENCE: {
        "per_receipt_limit": 1000.00,
        "daily_limit": 1000.00,
        "requires_attendees": False,
        "notes": "Pre-approval required. Registration + 1 tutorial covered.",
    },
    ExpenseCategory.COMMUNICATION: {
        "per_receipt_limit": 150.00,
        "daily_limit": 150.00,
        "requires_attendees": False,
        "notes": "Monthly phone/internet reimbursement.",
    },
    ExpenseCategory.TRANSPORTATION: {
        "per_receipt_limit": 50.00,
        "daily_limit": 100.00,
        "requires_attendees": False,
        "notes": "Public transit and parking.",
    },
    ExpenseCategory.CLIENT_ENTERTAINMENT: {
        "per_receipt_limit": 500.00,
        "daily_limit": 1000.00,
        "requires_attendees": True,
        "notes": "Client name and purpose required. Max $125/person.",
    },
    ExpenseCategory.MISCELLANEOUS: {
        "per_receipt_limit": 100.00,
        "daily_limit": 100.00,
        "requires_attendees": False,
        "notes": "Requires manager justification for > $50.",
    },
}


class PolicyValidator:
    """Agent 4: Validate receipts against company expense policy."""

    def __init__(self, policy: dict):
        self.policy = policy
        self._seen_ids: set[str] = set()
        self._call_log: list[dict] = []

    def validate(self, categorized: CategorizedReceipt) -> PolicyResult:
        """Check a categorized receipt against expense policy."""
        receipt = categorized.receipt
        category = categorized.category
        warnings: list[str] = []
        status = PolicyStatus.APPROVED
        approved_amount = receipt.total

        rules = self.policy.get(category, self.policy[ExpenseCategory.MISCELLANEOUS])

        # ── Duplicate check ──
        if receipt.receipt_id in self._seen_ids:
            return PolicyResult(
                status=PolicyStatus.DUPLICATE,
                warnings=["Duplicate receipt ID detected"],
                approved_amount=0.0,
                original_amount=receipt.total,
                policy_notes="Duplicate submission blocked.",
            )
        self._seen_ids.add(receipt.receipt_id)

        # ── Per-receipt limit ──
        limit = rules["per_receipt_limit"]
        if receipt.total > limit * 1.5:
            status = PolicyStatus.REJECTED
            approved_amount = limit
            warnings.append(f"Amount ${receipt.total:.2f} exceeds hard limit "
                           f"(${limit:.2f}) for {category.value} by "
                           f"${receipt.total - limit:.2f}")
        elif receipt.total > limit:
            status = PolicyStatus.WARNING
            approved_amount = receipt.total  # Still approved but flagged
            warnings.append(f"Amount ${receipt.total:.2f} exceeds soft limit "
                           f"(${limit:.2f}) for {category.value}")

        # ── Required fields ──
        if not receipt.receipt_date:
            warnings.append("Missing receipt date")
            if status == PolicyStatus.APPROVED:
                status = PolicyStatus.WARNING
        if receipt.total <= 0:
            status = PolicyStatus.REJECTED
            warnings.append("Total amount is zero or negative")
            approved_amount = 0.0

        # ── Low OCR confidence ──
        if receipt.confidence < 0.90:
            warnings.append(f"Low OCR confidence ({receipt.confidence:.0%}) — manual review recommended")
            if status == PolicyStatus.APPROVED:
                status = PolicyStatus.WARNING

        # ── Category-specific checks ──
        if rules.get("requires_attendees"):
            text_lower = receipt.raw_text.lower()
            if "client" not in text_lower and "attendee" not in text_lower:
                warnings.append(f"{category.value} requires client/attendee names")
                if status == PolicyStatus.APPROVED:
                    status = PolicyStatus.WARNING

        # ── Special: lodging per-night check ──
        if category == ExpenseCategory.LODGING:
            nights = receipt.raw_text.lower().count("night")
            if nights > 0:
                per_night = receipt.subtotal / max(nights, 1)
                if per_night > limit:
                    warnings.append(f"Per-night cost ${per_night:.2f} exceeds "
                                   f"${limit:.2f} limit")

        policy_notes = rules.get("notes", "")
        if not warnings:
            policy_notes = f"Within policy ({category.value}: max ${limit:.2f}/receipt). " + policy_notes

        result = PolicyResult(
            status=status,
            warnings=warnings,
            approved_amount=round(approved_amount, 2),
            original_amount=round(receipt.total, 2),
            policy_notes=policy_notes,
        )
        self._call_log.append({
            "receipt_id": receipt.receipt_id,
            "status": str(status),
            "original": receipt.total,
            "approved": approved_amount,
            "warnings": len(warnings),
        })
        return result

    @property
    def call_log(self):
        return self._call_log


validator = PolicyValidator(EXPENSE_POLICY)

# Demo
demo_policy = validator.validate(demo_cat)
print(f"Status:     {demo_policy.status}")
print(f"Original:   ${demo_policy.original_amount:.2f}")
print(f"Approved:   ${demo_policy.approved_amount:.2f}")
print(f"Warnings:   {demo_policy.warnings}")
print(f"Policy:     {demo_policy.policy_notes}")

## 8. Agent Chain — Full Pipeline

The `ReceiptExpenseAgent` wires all four agents into a single pipeline. Each receipt flows through all stages sequentially.

```text
for each receipt_id:
  raw_text, conf  = ocr_agent.process(receipt_id)
  receipt_data    = extractor.extract(receipt_id, raw_text, conf)
  categorized     = categorizer.categorize(receipt_data)
  policy_result   = validator.validate(categorized)
  → ProcessedReceipt(receipt_data, category, policy_result)
```


In [ ]:
class ReceiptExpenseAgent:
    """End-to-end receipt processing agent chain."""

    def __init__(self):
        self.ocr = OCRAgent()
        self.extractor = FieldExtractor()
        self.categorizer = ExpenseCategorizer()
        self.validator = PolicyValidator(EXPENSE_POLICY)
        self._results: list[ProcessedReceipt] = []

    def process_receipt(self, receipt_id: str,
                        verbose: bool = False) -> ProcessedReceipt:
        """Process a single receipt through the full agent chain."""

        # Stage 1: OCR
        raw_text, confidence = self.ocr.process(receipt_id)
        if verbose:
            vendor_line = raw_text.strip().split("\n")[0] if raw_text else "N/A"
            print(f"[OCR]       {receipt_id} → {vendor_line[:40]}  conf={confidence:.2f}")

        if not raw_text:
            empty = ReceiptData(
                receipt_id=receipt_id, vendor_name="OCR_FAILED",
                vendor_address="", receipt_date="", payment_method="",
                currency="USD", subtotal=0, tax=0, tip=0, total=0,
                items=[], raw_text="", confidence=0.0,
            )
            return ProcessedReceipt(
                receipt=empty,
                category=ExpenseCategory.MISCELLANEOUS,
                category_confidence=0.0,
                policy=PolicyResult(PolicyStatus.REJECTED, ["OCR failed"], 0, 0, ""),
            )

        # Stage 2: Field extraction
        receipt = self.extractor.extract(receipt_id, raw_text, confidence)
        if verbose:
            print(f"[EXTRACT]   {receipt.vendor_name:<30} "
                  f"${receipt.total:>8.2f}  items={len(receipt.items)}")

        # Stage 3: Categorization
        categorized = self.categorizer.categorize(receipt)
        if verbose:
            print(f"[CATEGORY]  {categorized.category.value:<20} "
                  f"conf={categorized.category_confidence:.2f}  "
                  f"({categorized.category_reason[:50]})")

        # Stage 4: Policy validation
        policy = self.validator.validate(categorized)
        if verbose:
            status_icon = {"approved": "OK", "warning": "!!", "rejected": "XX",
                          "duplicate": "DUP"}.get(str(policy.status), "??")
            print(f"[POLICY]    [{status_icon}] ${policy.approved_amount:.2f} / "
                  f"${policy.original_amount:.2f}  "
                  f"{'  '.join(policy.warnings) if policy.warnings else 'clean'}")

        result = ProcessedReceipt(
            receipt=receipt,
            category=categorized.category,
            category_confidence=categorized.category_confidence,
            policy=policy,
        )
        self._results.append(result)
        return result

    def process_batch(self, receipt_ids: list[str],
                      verbose: bool = False) -> list[ProcessedReceipt]:
        """Process multiple receipts."""
        results = []
        for rid in receipt_ids:
            results.append(self.process_receipt(rid, verbose=verbose))
            if verbose:
                print()
        return results

    def generate_report(self, employee_name: str) -> ReimbursementReport:
        """Generate a reimbursement summary report."""
        total_submitted = sum(r.policy.original_amount for r in self._results)
        total_approved = sum(
            r.policy.approved_amount for r in self._results
            if r.policy.status in (PolicyStatus.APPROVED, PolicyStatus.WARNING)
        )
        total_rejected = sum(
            r.policy.original_amount for r in self._results
            if r.policy.status in (PolicyStatus.REJECTED, PolicyStatus.DUPLICATE)
        )

        cat_totals: dict[str, float] = defaultdict(float)
        for r in self._results:
            if r.policy.status in (PolicyStatus.APPROVED, PolicyStatus.WARNING):
                cat_totals[str(r.category)] += r.policy.approved_amount

        all_warnings = []
        for r in self._results:
            for w in r.policy.warnings:
                all_warnings.append(f"{r.receipt.receipt_id}: {w}")

        # Determine overall approval
        rejected_count = sum(1 for r in self._results
                            if r.policy.status == PolicyStatus.REJECTED)
        warning_count = sum(1 for r in self._results
                           if r.policy.status == PolicyStatus.WARNING)
        if rejected_count > 0:
            approval = "needs_review"
        elif warning_count > 0:
            approval = "needs_review"
        else:
            approval = "auto_approved"

        return ReimbursementReport(
            employee_name=employee_name,
            submission_date=datetime.now().strftime("%Y-%m-%d"),
            receipts=self._results,
            total_submitted=round(total_submitted, 2),
            total_approved=round(total_approved, 2),
            total_rejected=round(total_rejected, 2),
            category_totals=dict(cat_totals),
            warnings=all_warnings,
            approval_status=approval,
        )

    @property
    def results(self):
        return self._results


agent = ReceiptExpenseAgent()
print("ReceiptExpenseAgent ready — 4-stage chain.")

## 9. Processing All Receipts

In [ ]:
all_ids = list(RECEIPT_TEXTS.keys())
results = agent.process_batch(all_ids, verbose=True)
print(f"Processed {len(results)} receipts.")

## 10. Results Overview

In [ ]:
rows = []
for r in agent.results:
    rows.append({
        "receipt_id": r.receipt.receipt_id,
        "vendor": r.receipt.vendor_name[:25],
        "date": r.receipt.receipt_date,
        "category": str(r.category),
        "original": r.policy.original_amount,
        "approved": r.policy.approved_amount,
        "status": str(r.policy.status),
        "warnings": len(r.policy.warnings),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

## 11. Reimbursement Report

In [ ]:
report = agent.generate_report("Sarah Johnson")

print("=" * 70)
print("  EXPENSE REIMBURSEMENT REPORT")
print("=" * 70)
print(f"  Employee:         {report.employee_name}")
print(f"  Submission Date:  {report.submission_date}")
print(f"  Approval Status:  {report.approval_status.upper()}")
print(f"  Receipts:         {len(report.receipts)}")
print()
print(f"  Total Submitted:  ${report.total_submitted:>10,.2f}")
print(f"  Total Approved:   ${report.total_approved:>10,.2f}")
print(f"  Total Rejected:   ${report.total_rejected:>10,.2f}")
print(f"  Savings:          ${report.total_submitted - report.total_approved:>10,.2f}")
print()
print("  Category Breakdown:")
for cat, total in sorted(report.category_totals.items(), key=lambda x: -x[1]):
    pct = total / report.total_approved * 100 if report.total_approved else 0
    bar = "#" * int(pct / 2)
    print(f"    {cat:<25} ${total:>9,.2f}  ({pct:4.1f}%)  {bar}")
print()
if report.warnings:
    print("  Warnings / Policy Flags:")
    for w in report.warnings:
        print(f"    ! {w}")
print("=" * 70)

## 12. Verbose Trace — Single Receipt (Client Dinner)

In [ ]:
# Re-run the most complex receipt with fresh agents to see full trace
trace_agent = ReceiptExpenseAgent()

print("FULL AGENT-CHAIN TRACE")
print("=" * 70)
result = trace_agent.process_receipt("REC-009", verbose=True)
print(f"\n{'='*70}")
print(f"Final Output:")
print(f"  Vendor:     {result.receipt.vendor_name}")
print(f"  Date:       {result.receipt.receipt_date}")
print(f"  Category:   {result.category}")
print(f"  Amount:     ${result.receipt.total:.2f}")
print(f"  Status:     {result.policy.status}")
print(f"  Approved:   ${result.policy.approved_amount:.2f}")
print(f"\n  Items extracted:")
for item in result.receipt.items:
    print(f"    {item.description:<25} {item.quantity} x ${item.unit_price:.2f} = ${item.total:.2f}")
print(f"\n  Policy notes: {result.policy.policy_notes}")
if result.policy.warnings:
    print(f"  Warnings:")
    for w in result.policy.warnings:
        print(f"    ! {w}")

## 13. OCR + Agent Chaining — Deep Dive

### How OCR Fits Into the Agent Chain

OCR is the **perception layer** — it converts an unstructured image into text that downstream agents can process. The key challenge is that OCR output is **noisy and variable**:

```text
Real OCR challenges:
┌──────────────────────────────────┐
│ • Skewed/rotated receipt images  │ → Deskew preprocessing
│ • Faded thermal paper            │ → Contrast enhancement
│ • Multi-column layouts           │ → Layout analysis (boxes)
│ • Handwritten tips/notes         │ → Handwriting recognition
│ • Foreign currencies/languages   │ → Multi-language support
│ • Crumpled/folded receipts       │ → Image reconstruction
└──────────────────────────────────┘
```

### Agent Chaining Pattern

Each agent is **stateless** and **composable**. The output of agent N is the input of agent N+1.

```python
# The chain is explicitly wired — no hidden state
raw_text  = ocr.process(image)           # str
receipt   = extractor.extract(raw_text)  # ReceiptData
categorized = categorizer.categorize(receipt)  # CategorizedReceipt
policy    = validator.validate(categorized)    # PolicyResult
```

### Benefits of Explicit Chaining

1. **Each agent can be retried independently** — if OCR fails, retry OCR without re-running extraction
2. **Logging per stage** — know exactly which agent produced a warning
3. **Partial results are useful** — extraction works even if categorization fails
4. **Agents can be LLM or rule-based** — swap implementations without changing the chain
5. **Testing is straightforward** — unit test each agent with known inputs

### Production OCR Pipeline

```python
# Production pipeline with real OCR:
class ProductionOCR:
    def __init__(self, engine="azure"):
        if engine == "azure":
            from azure.ai.formrecognizer import DocumentAnalysisClient
            self.client = DocumentAnalysisClient(endpoint, credential)
        elif engine == "tesseract":
            import pytesseract
            self.engine = pytesseract

    def process(self, image_bytes: bytes) -> tuple[str, float]:
        # Azure Document Intelligence
        poller = self.client.begin_analyze_document(
            "prebuilt-receipt", image_bytes
        )
        result = poller.result()
        # Extract fields directly (Azure does extraction too)
        return result.content, result.confidence
```

### When to Use LLMs in the Chain

| Stage | Rule-based | LLM-based | When to upgrade |
|---|---|---|---|
| OCR | Tesseract | GPT-4V, Claude Vision | Handwritten/damaged receipts |
| Extraction | Regex | Structured output mode | Non-standard layouts |
| Categorization | Keyword rules | Classification prompt | Ambiguous expenses |
| Policy check | Threshold rules | Keep rule-based | Never — policy must be deterministic |
| Summary | Template | LLM generation | Natural-language summaries needed |


## 14. Expense Analytics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Chart 1: Category breakdown (pie) ──
ax = axes[0, 0]
cat_data = results_df.groupby("category")["approved"].sum().sort_values(ascending=False)
cat_data = cat_data[cat_data > 0]
colors_map = {
    "meals": "#ff9f43", "travel": "#54a0ff", "lodging": "#5f27cd",
    "software": "#10ac84", "office_supplies": "#ee5a24",
    "conference": "#0abde3", "communication": "#c8d6e5",
    "transportation": "#feca57", "client_entertainment": "#ff6b6b",
    "miscellaneous": "#95afc0",
}
cat_colors = [colors_map.get(c, "#95afc0") for c in cat_data.index]
wedges, texts, autotexts = ax.pie(
    cat_data.values, labels=cat_data.index, autopct="%1.0f%%",
    colors=cat_colors, startangle=90, textprops={"fontsize": 8},
)
ax.set_title("Approved Expenses by Category")

# ── Chart 2: Policy status breakdown ──
ax2 = axes[0, 1]
status_counts = results_df["status"].value_counts()
status_colors = {"approved": "#27ae60", "warning": "#f39c12",
                 "rejected": "#e74c3c", "duplicate": "#95a5a6"}
bars = ax2.bar(status_counts.index, status_counts.values,
               color=[status_colors.get(s, "#95a5a6") for s in status_counts.index],
               alpha=0.8, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, status_counts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
             str(val), ha="center", fontsize=11, fontweight="bold")
ax2.set_ylabel("Count")
ax2.set_title("Receipts by Policy Status")

# ── Chart 3: Original vs Approved amounts ──
ax3 = axes[1, 0]
x_pos = range(len(results_df))
ax3.bar(x_pos, results_df["original"], alpha=0.4, color="#3498db",
        label="Submitted", edgecolor="black", linewidth=0.3)
ax3.bar(x_pos, results_df["approved"], alpha=0.8, color="#27ae60",
        label="Approved", edgecolor="black", linewidth=0.3)
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels(results_df["receipt_id"], rotation=45, ha="right", fontsize=8)
ax3.set_ylabel("Amount ($)")
ax3.set_title("Submitted vs. Approved Amount")
ax3.legend(fontsize=8)

# ── Chart 4: OCR confidence vs extraction success ──
ax4 = axes[1, 1]
conf_data = [(r.receipt.confidence, len(r.receipt.items),
              str(r.policy.status)) for r in agent.results]
confs = [c[0] for c in conf_data]
items = [c[1] for c in conf_data]
statuses = [c[2] for c in conf_data]
status_c = [status_colors.get(s, "#95a5a6") for s in statuses]
ax4.scatter(confs, items, c=status_c, s=120, edgecolors="black",
            linewidth=0.5, alpha=0.8, zorder=3)
for i, r in enumerate(agent.results):
    ax4.annotate(r.receipt.receipt_id[-3:], (confs[i], items[i]),
                 fontsize=7, ha="left", va="bottom")
ax4.set_xlabel("OCR Confidence")
ax4.set_ylabel("Items Extracted")
ax4.set_title("OCR Confidence vs. Items Extracted")
# Legend
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in status_colors.items()]
ax4.legend(handles=legend_patches, fontsize=7, loc="upper left")
ax4.axvline(0.95, color="red", linestyle=":", alpha=0.5, label="High-conf threshold")

plt.suptitle("Receipt Expense Agent — Analytics Dashboard",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 15. Agent Call Log & Pipeline Metrics

In [ ]:
print("PIPELINE CALL LOG")
print("=" * 60)
print(f"  OCR calls:          {len(agent.ocr.call_log)}")
print(f"  Extractor calls:    {len(agent.extractor.call_log)}")
print(f"  Categorizer calls:  {len(agent.categorizer.call_log)}")
print(f"  Validator calls:    {len(agent.validator.call_log)}")
print()

# OCR success rate
ocr_success = sum(1 for c in agent.ocr.call_log if c["success"])
print(f"  OCR success rate:   {ocr_success}/{len(agent.ocr.call_log)}")

# Avg OCR confidence
avg_conf = np.mean([c.get("confidence", 0) for c in agent.ocr.call_log if c["success"]])
print(f"  Avg OCR confidence: {avg_conf:.2f}")

# Items extracted
total_items = sum(c["items_found"] for c in agent.extractor.call_log)
print(f"  Total items found:  {total_items}")

# Category distribution
cat_dist = Counter(c["category"] for c in agent.categorizer.call_log)
print(f"\n  Category distribution:")
for cat, count in cat_dist.most_common():
    print(f"    {cat:<25} {count}")

# Policy outcomes
pol_dist = Counter(c["status"] for c in agent.validator.call_log)
print(f"\n  Policy outcomes:")
for status, count in pol_dist.most_common():
    print(f"    {status:<15} {count}")

In [ ]:
# Pipeline flow visualization
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

stages = ["OCR\nAgent", "Field\nExtractor", "Expense\nCategorizer",
          "Policy\nValidator", "Processed\nReceipt"]
stage_counts = [len(agent.ocr.call_log),
                len(agent.extractor.call_log),
                len(agent.categorizer.call_log),
                len(agent.validator.call_log),
                len(agent.results)]

stage_colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"]

# Draw flow boxes
box_width = 0.12
for i, (stage, count, color) in enumerate(zip(stages, stage_counts, stage_colors)):
    x = i * 0.2 + 0.1
    rect = mpatches.FancyBboxPatch((x - box_width/2, 0.3), box_width, 0.4,
                                    boxstyle="round,pad=0.02",
                                    facecolor=color, alpha=0.7,
                                    edgecolor="black", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 0.5, f"{stage}\n({count})", ha="center", va="center",
            fontsize=9, fontweight="bold", color="white")

    # Draw arrows
    if i < len(stages) - 1:
        ax.annotate("", xy=(x + box_width/2 + 0.04, 0.5),
                    xytext=(x + box_width/2 + 0.01, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=2, color="#2c3e50"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_axis_off()
ax.set_title("Agent Chain — Pipeline Flow", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

## 16. Evaluation

In [ ]:
EXPECTED = {
    "REC-001": {"category": "meals",               "status_in": ["approved", "warning"]},
    "REC-002": {"category": "travel",              "status_in": ["approved"]},
    "REC-003": {"category": "lodging",             "status_in": ["warning", "rejected"]},
    "REC-004": {"category": "software",            "status_in": ["approved"]},
    "REC-005": {"category": "office_supplies",     "status_in": ["approved"]},
    "REC-006": {"category": "meals",               "status_in": ["approved"]},
    "REC-007": {"category": "conference",          "status_in": ["approved"]},
    "REC-008": {"category": "communication",       "status_in": ["approved"]},
    "REC-009": {"category": "client_entertainment","status_in": ["rejected", "warning"]},
    "REC-010": {"category": "transportation",      "status_in": ["approved"]},
}

eval_rows = []
for r in agent.results:
    rid = r.receipt.receipt_id
    exp = EXPECTED.get(rid, {})

    cat_ok = str(r.category) == exp.get("category", str(r.category))
    status_ok = str(r.policy.status) in exp.get("status_in", [str(r.policy.status)])
    has_vendor = bool(r.receipt.vendor_name and r.receipt.vendor_name != "Unknown Vendor")
    has_date = bool(r.receipt.receipt_date)
    has_total = r.receipt.total > 0
    has_category = r.category_confidence > 0.5
    passed = all([cat_ok, status_ok, has_vendor, has_date, has_total, has_category])

    eval_rows.append({
        "receipt": rid,
        "cat_ok": cat_ok,
        "status_ok": status_ok,
        "vendor_ok": has_vendor,
        "date_ok": has_date,
        "total_ok": has_total,
        "conf_ok": has_category,
        "passed": passed,
    })

eval_df = pd.DataFrame(eval_rows)
print("EVALUATION RESULTS")
print("=" * 80)
for _, row in eval_df.iterrows():
    checks = " ".join(f"{k}={'ok' if v else 'FAIL'}" for k, v in row.items()
                       if k not in ("receipt", "passed"))
    icon = "PASS" if row["passed"] else "FAIL"
    print(f"  [{icon}] {row['receipt']}  {checks}")

print(f"\nOverall: {eval_df['passed'].sum()}/{len(eval_df)} "
      f"({eval_df['passed'].mean():.0%})")

## 17. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "receipt_expense_agent",
    "receipts_processed": len(agent.results),
    "total_submitted": report.total_submitted,
    "total_approved": report.total_approved,
    "total_rejected": report.total_rejected,
    "approval_status": report.approval_status,
    "category_totals": report.category_totals,
    "pipeline_stages": {
        "ocr_agent": "Simulated OCR with confidence scoring",
        "field_extractor": "Regex-based structured field extraction",
        "expense_categorizer": "Keyword-rule classification (10 categories)",
        "policy_validator": "Per-category limit + duplicate + field checks",
        "summary_generator": "Aggregated reimbursement report",
    },
    "eval_pass_rate": float(eval_df["passed"].mean()),
}

log_path = ARTIFACT_DIR / "receipt_expense_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")

receipt_log = []
for r in agent.results:
    receipt_log.append({
        "receipt_id": r.receipt.receipt_id,
        "vendor": r.receipt.vendor_name,
        "date": r.receipt.receipt_date,
        "total": r.receipt.total,
        "category": str(r.category),
        "status": str(r.policy.status),
        "approved": r.policy.approved_amount,
        "warnings": r.policy.warnings,
    })

receipts_path = ARTIFACT_DIR / "processed_receipts.json"
receipts_path.write_text(json.dumps(receipt_log, indent=2), encoding="utf-8")

print(f"Saved: {log_path}")
print(f"Saved: {receipts_path}")

## 18. Key Takeaways

### What We Built
- A **4-agent chain** for end-to-end receipt expense processing
- **10 realistic receipts** spanning 8 expense categories
- **OCR agent** with confidence scoring and noise simulation
- **Regex-based field extraction** with multi-format date parsing and item-line detection
- **Keyword-based categorization** into 10 company expense categories
- **Policy validation** with per-category limits, duplicate detection, and field completeness checks
- **Reimbursement report** with category breakdown, flags, and approval status

### OCR + Agent Chaining Principles

| Principle | Application |
|---|---|
| **Separation of concerns** | OCR only produces text; extraction only parses fields; categorizer only classifies |
| **Typed interfaces** | Each agent returns a dataclass, not a raw dict |
| **Graceful degradation** | Low OCR confidence → warning flag, not crash |
| **Independent testability** | Test field extraction with known text strings |
| **Swappable backends** | Replace simulated OCR with Azure Document Intelligence |
| **Deterministic policy** | Policy validation is rule-based (not LLM) for auditability |
| **Call logging** | Every agent logs its calls for observability |

### Production Enhancements

| Enhancement | Purpose |
|---|---|
| **Azure Form Recognizer** | Replace simulated OCR with real document AI |
| **LLM extraction** | GPT-4o with structured output for complex receipts |
| **Receipt deduplication** | Fuzzy matching on vendor + date + amount |
| **Multi-currency** | Auto-convert foreign receipts to base currency |
| **Manager approval workflow** | Route flagged receipts to manager for sign-off |
| **ERP integration** | Push approved expenses to SAP / NetSuite |
| **Mobile upload** | Camera capture → instant OCR → draft expense |
| **Annual spend analytics** | Dashboard showing spending patterns over time |
